# Example 5.7

In [ ]:
# main script
import numpy as np


def Ej_5_7(N, Tw):
    """
    Example 5.7

    Input variables:
    N  : number of spatial grid elements
    Tw : wall temperature at x = 1

    Output variables:
    Output1 : Explicit Euler results (Eq. 5.33), shape (3,5)
    Output2 : Implicit Euler results (Eq. 5.35), shape (3,5)
    Output3 : Crank–Nicolson results (Eq. 5.37), shape (3,5)
    """

    deltasT = np.array([0.1, 0.25, 0.5])      # time integration steps
    taus = np.array([0.5, 1, 2, 3, 4])        # integration times

    Output1 = np.zeros((3, 5))
    Output2 = np.zeros((3, 5))
    Output3 = np.zeros((3, 5))

    for i in range(3):
        dt = deltasT[i]

        # Constant vectors defining the tridiagonal systems
        a = dt * np.ones(N - 1)
        b = -(1 + 2 * dt) * np.ones(N - 1)
        c = dt * np.ones(N - 1)

        d = np.ones(N - 1)
        e = -2 * (1 + 1 / dt) * d
        f = d

        # ---------- Explicit Euler method ----------
        t0 = 0.0
        y0 = np.zeros(N + 1)
        y0[-1] = Tw

        for j in range(5):
            tf = taus[j]
            Tout = fun1(N, dt, t0, tf, y0)
            Output1[i, j] = Tout[N - 1]
            y0 = Tout
            t0 = tf

        # ---------- Implicit Euler method ----------
        t0 = 0.0
        y0 = np.zeros(N + 1)
        y0[-1] = Tw

        for j in range(5):
            tf = taus[j]
            Tout = fun2(N, dt, t0, tf, y0, Tw, a.copy(), b.copy(), c.copy())
            Output2[i, j] = Tout[N - 1]
            y0 = Tout
            t0 = tf

        # ---------- Crank–Nicolson method ----------
        t0 = 0.0
        y0 = np.zeros(N + 1)
        y0[-1] = Tw

        for j in range(5):
            tf = taus[j]
            Tout = fun3(N, dt, t0, tf, y0, Tw, d.copy(), e.copy(), f.copy())
            Output3[i, j] = Tout[N - 1]
            y0 = Tout
            t0 = tf

    return Output1, Output2, Output3

def fun1(N, dt, t0, tf, y0):
    """Explicit Euler method"""
    y = y0.copy()
    M = int((tf - t0) / dt)

    for _ in range(M):
        y_new = y.copy()
        for i in range(1, N):
            y_new[i] = dt * (y[i - 1] + y[i + 1]) + (1 - 2 * dt) * y[i]
        y = y_new

    return y

def fun2(N, dt, t0, tf, y0, Tw, a, b, c):
    """Implicit Euler (Backward Euler) method"""
    y = y0.copy()
    M = int((tf - t0) / dt)

    for _ in range(M):
        r = -y[1:N]
        r[-1] -= dt * Tw
        v = tridisolve(a, b, c, r)
        y = np.concatenate(([y[0]], v, [y[-1]]))

    return y

def fun3(N, dt, t0, tf, y0, Tw, d, e, f):
    """Crank–Nicolson method"""
    y = y0.copy()
    M = int((tf - t0) / dt)

    for _ in range(M):
        r = np.zeros(N - 1)
        for i in range(N - 1):
            r[i] = -y[i] + 2 * (1 - 1 / dt) * y[i + 1] - y[i + 2]
        r[-1] -= Tw

        v = tridisolve(d, e, f, r)
        y = np.concatenate(([y[0]], v, [y[-1]]))

    return y

def tridisolve(a, b, c, d):
    """
    Solves a tridiagonal system using the Thomas algorithm
    """
    n = len(d)
    b = b.copy()
    x = d.copy()

    for j in range(n - 1):
        mu = a[j] / b[j]
        b[j + 1] -= mu * c[j]
        x[j + 1] -= mu * x[j]

    x[-1] /= b[-1]
    for j in range(n - 2, -1, -1):
        x[j] = (x[j] - c[j] * x[j + 1]) / b[j]

    return x

In [ ]:
# script for generating figure 5.10

import numpy as np
import matplotlib.pyplot as plt

# Run Example 5.7
Output1, Output2, Output3 = Ej_5_7(10, 10)

dT   = [0.1, 0.25, 0.5]          # time steps
taus = np.array([0.5, 1, 2, 3, 4])

fig, axs = plt.subplots(2, 2, figsize=(20/2.54, 15/2.54))

# --- Explicit Euler ---
axs[0,0].plot(taus, Output1[0,:], '-k', label=r'$\Delta\tau=0.1$')
axs[0,0].plot(taus, Output1[1,:], 'k+', label=r'$\Delta\tau=0.25$')
axs[0,0].plot(taus, Output1[2,:], 'ko', label=r'$\Delta\tau=0.5$')
axs[0,0].set_xlabel(r'time, $\tau$')
axs[0,0].set_ylabel(r'$y(x=0.9,\tau)$')
axs[0,0].set_title('Explicit Euler')
axs[0,0].legend(loc='lower right')

# --- Implicit Euler ---
axs[0,1].plot(taus, Output2[0,:], '-k')
axs[0,1].plot(taus, Output2[1,:], 'k+')
axs[0,1].plot(taus, Output2[2,:], 'ko')
axs[0,1].set_xlabel(r'time, $\tau$')
axs[0,1].set_ylabel(r'$y(x=0.9,\tau)$')
axs[0,1].set_title('Implicit Euler')
axs[0,1].legend([r'$\Delta\tau=0.1$', r'$\Delta\tau=0.25$', r'$\Delta\tau=0.5$'],
                loc='lower right')

# --- Crank–Nicolson ---
axs[1,0].plot(taus, Output3[0,:], '-k')
axs[1,0].plot(taus, Output3[1,:], 'k+')
axs[1,0].plot(taus, Output3[2,:], 'ko')
axs[1,0].set_xlabel(r'time, $\tau$')
axs[1,0].set_ylabel(r'$y(x=0.9,\tau)$')
axs[1,0].set_title('Crank–Nicolson')
axs[1,0].legend([r'$\Delta\tau=0.1$', r'$\Delta\tau=0.25$', r'$\Delta\tau=0.5$'],
                loc='lower right')

# --- Comparison of methods ---
axs[1,1].plot(taus, Output1[0,:], '-k', label=r'Exp., $\Delta\tau=0.1$')
axs[1,1].plot(taus, Output2[1,:], 'k+', label=r'Imp., $\Delta\tau=0.25$')
axs[1,1].plot(taus, Output3[2,:], 'ko', label=r'C–N, $\Delta\tau=0.5$')
axs[1,1].set_xlabel(r'time, $\tau$')
axs[1,1].set_ylabel(r'$y(x=0.9,\tau)$')
axs[1,1].set_title('Comparison of methods')
axs[1,1].legend(loc='lower right')

plt.tight_layout()
filename = 'Figure_5_10.png'
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()